# exp-01 — detailed skill instructions against minimal ones

Reads the runs committed under `experiments/runs/exp-01-skill-verbosity/`,
scores every leaf that is not scored yet, and compares the two arms.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-01-skill-verbosity/run.py
```

which is thousands of sessions and hours long. Keeping the run out of here
is what makes the analysis re-executable: anyone can rerun this against the
committed predictions without paying for them again.

Scoring *is* done here, because the scored `analysis.json` is derived data
and deliberately not committed — it is a pure function of the committed
predictions, the committed gold and the committed `eval-manifest.json`. It
is cheap and local, and the cell below skips any leaf already scored.

The hypothesis, the decision criteria and the findings live in
[`thinking/experiments/exp-01-skill-verbosity.md`](../../thinking/experiments/exp-01-skill-verbosity.md).
Read that first — this notebook is the arithmetic behind it, not the claim.

## The arms

Each check carries two versions of its own skill, and the harness names them
by what moved off the manifest pin:

| directory | arm | skill |
|---|---|---|
| `pinned` | **detailed** | `v1`, full stepwise instructions |
| `<check>@v2` | **minimal** | `v2`, same frontmatter and opening paragraph, nothing else |

Both are scored by the same `schema.json`, the same `eval-manifest.json` and
the same gold: the contracts sit above the version directories, so the
control is structural rather than asserted.

In [ ]:
from pathlib import Path

import pandas as pd

from soda_mmqc.core.run_layout import BASELINE_ARM, iter_leaves
from soda_mmqc.reporting.styles import LAYER1_ORDER, LAYER_S_ORDER
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import (
    arm_contrast,
    arm_levels,
    layer1_counts,
    layer_s_counts,
    plot_arm_contrast_by_check,
    plot_arm_levels_by_check,
    plot_check_layers,
    plot_grouped_counts,
    plot_stacked_counts,
    load_run_root,
    non_response_counts,
    replicate_spread,
    scores_frame,
)

# Property paths run past pandas' 50-character default, and a label
# truncated to `...outputs[].annotati...` is the problem this column
# exists to fix.
pd.set_option("display.max_colwidth", None)
# The contrast tables run to 68 rows and pandas shows ten of a frame
# that long. These tables are the result, not a preview of it.
pd.set_option("display.max_rows", 200)

RUNS = Path("../../experiments/runs/exp-01-skill-verbosity").resolve()
CHECKLIST = "fig-checklist-exp01"
# Fixed once, so the two arms keep the same colour in every figure.
ARM_ORDER = ("detailed", "minimal")

assert RUNS.is_dir(), f"no runs at {RUNS}; produce them with run.py first"
checks = sorted(p.name for p in RUNS.iterdir() if p.is_dir())
checks

## Score every leaf that is not scored yet

A leaf is one `<check>/<arm>/rep-NN/` directory, and `score_check` takes
exactly one. Aggregating across arms and replicates is this notebook's job,
not the harness's — which is what keeps the evaluator general.

This is idempotent: a leaf that already has an `analysis.json` is skipped,
so re-running the notebook costs nothing after the first time.

In [ ]:
for check in checks:
    for arm, replicate, leaf in iter_leaves(RUNS / check):
        if (leaf / ANALYSIS_FILENAME).is_file():
            continue
        score_check(CHECKLIST, check, leaf, save=True)
        print(f"scored {check} / {arm} / rep-{replicate:02d}")
print("all leaves scored")

## One row per property, per replicate, per example

`scores_frame` keeps both axes of variation — which example, which
replicate — and never collapses `property` into a row. Every statistic
below groups this frame, and none of them group across `property`.

In [ ]:
frames = []
for check in checks:
    runs = load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
    frames.append(scores_frame(runs))

scores = pd.concat(frames, ignore_index=True)


def arm_label(arm: str) -> str:
    """`pinned` is the detailed arm; anything else moved a skill to v2."""
    return "detailed" if arm == BASELINE_ARM else "minimal"


scores["arm_label"] = scores["arm"].map(arm_label)
print(f"{len(scores)} rows, {scores['check'].nunique()} checks, "
      f"{scores['replicate'].nunique()} replicates")
scores.head()

## 1. Non-response, first

Read this before any mean. A session that answers `outputs: []` scores as a
fully missing row set rather than being excluded — a skill so thin the model
returns nothing is a worse result, not an absent one.

It also interacts with layer 2 in the direction that misleads: an arm that
answers nothing has no applicable instances either, so it contributes
nothing to the layer-2 mean and can appear to score **better** by saying
less. A difference in non-response *is* part of the result.

In [ ]:
non_response = pd.concat(
    [
        non_response_counts(
            load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
        )
        for check in checks
    ],
    ignore_index=True,
)
non_response["arm_label"] = non_response["arm"].map(arm_label)

(
    non_response.groupby(["check", "arm_label"])["empty"]
    .sum()
    .unstack(fill_value=0)
)

## 2. Applicability (layer 1), second

Layer 2 is conditional on layer 1: a property's mean is taken over the
instances the model judged applicable *and* judged so correctly. So the
number of instances each arm was scored on is itself a result, and it is
what makes the layer-2 means comparable or not.

`n_scored` is that denominator. Where the two arms differ materially here,
the layer-2 comparison below is between different populations.

In [ ]:
denominators = (
    scores.groupby(["check", "property", "arm_label"])["n_scored"]
    .sum()
    .unstack(fill_value=0)
)
denominators["ratio"] = (
    denominators["minimal"] / denominators["detailed"].replace(0, pd.NA)
)
denominators.sort_values("ratio").head(15)

## 3. Spread across replicates

How much would a number move if we ran it again? This is the SD over
replicates, per arm and per property — examples are collapsed within a
replicate first, so this is resampling noise and not example-to-example
variation.

A replicate that scored nothing applicable is excluded, not counted as
zero. `sd` is NA below two replicates, because the SD of one observation is
undefined rather than zero.

In [ ]:
spread = pd.concat(
    [
        replicate_spread(scores[scores["check"] == check])
        for check in checks
    ],
    ignore_index=True,
)
spread["arm_label"] = spread["arm"].map(arm_label)
(
    spread.sort_values("sd", ascending=False)
    .set_index(["check", "arm_label", "property"])
    [["mean", "sd", "n_replicates", "n_scored_total"]]
    .head(15)
)

## 4. The contrast, per property

**The earlier version of this notebook averaged `mean_score` across
properties.** That number mixes `panel_label` with `micrograph`, so an arm
that improves one and degrades the other looks unchanged. The comparison
below is per property, and there is no headline number — deliberately.

`difference` is `minimal − detailed`, paired by example within a property.
Replicates are averaged per `(arm, example)` first: they are resamples of
one measurement, so they reduce its noise rather than adding rows.

**Read `paired_fraction` first.** Each arm's mean is over its own applicable
set, so a difference is computed only on the examples *both* arms scored. An
arm that answers less is scored on fewer, self-selected cases, and a
difference of zero over one of forty examples is not the same finding as a
difference of zero over forty.

Rows are indexed by `(check, property)`. `outputs[].panel_label` alone is
ambiguous -- it occurs in all eleven checks -- and the check has to be on
the row for the table to mean anything. Sorting is by the column that
each table is about, so a check appears wherever its properties fall
rather than in one block.

The frames also carry a flat `path` column,
`check:arm:replicate:example:property` with an empty segment where an
axis was pooled, which reads `micrograph-scale-bar::::outputs[].panel_label`
here. It is one copyable token for a single measurement, and it is what
the figure below shows on hover.

In [ ]:
contrasts = []
for check in checks:
    per_check = scores[scores["check"] == check]
    variants = sorted(set(per_check["arm"]) - {BASELINE_ARM})
    for variant in variants:
        contrast = arm_contrast(
            per_check, baseline=BASELINE_ARM, variant=variant
        )
        contrasts.append(contrast)

contrast = pd.concat(contrasts, ignore_index=True)
contrast = contrast.sort_values("paired_fraction")
contrast.set_index(["check", "property"])[[
    "difference", "se",
    "n_examples", "n_baseline_only", "n_variant_only", "paired_fraction",
]]

### Where the pairing held

Rows with `paired_fraction` near 1 are the ones where the two arms were
scored on the same examples, so the difference means what it looks like.
Rows well below 1 are qualified by construction.

In [ ]:
solid = contrast[contrast["paired_fraction"] >= 0.9].copy()
solid = solid.sort_values("difference")
print(f"{len(solid)} of {len(contrast)} property contrasts on a near-complete pairing")
solid.set_index(["check", "property"])[["difference", "se", "n_examples"]]

### The differences, with their standard errors

One panel per check, one bar per property, and a shared x axis so a small
difference in one check cannot be mistaken for a large one in another. The
panel title carries the check, so a tick keeps only the within-record path.

A difference is only as good as its `se` and its pairing, both of which are
on the row beside it in the table above. The bars are one neutral colour on
purpose: which direction counts as better is the claim, and the claim lives
in the note.

In [ ]:
plot_arm_contrast_by_check(
    solid,
    title="minimal − detailed, per property (near-complete pairings only)",
)

### Where on the scale that happened

The same contrasts as the figure above, as absolute scores. A difference of
−0.05 is a different finding at 0.95 than at 0.20, and the difference alone
cannot tell them apart.

Both bars are means over the **paired** examples — the same intersection
`arm_contrast` uses — so the gap between a pair *is* the difference plotted
above. Each arm's mean over its own examples would not be: on this run it
disagreed for 17 of 67 contrasts, by as much as 0.14, and reversed the sign
of `stat-test`'s `explanation`. Two figures side by side that contradict
each other are worse than one.

The axis spans the whole of `mean_score` rather than the data, so a gap
between 0.88 and 0.90 is drawn as the small thing it is.

In [ ]:
levels = pd.concat(
    [
        arm_levels(
            scores[scores["check"] == check],
            baseline=BASELINE_ARM,
            variant=variant,
        )
        for check in checks
        for variant in sorted(
            set(scores[scores["check"] == check]["arm"]) - {BASELINE_ARM}
        )
    ],
    ignore_index=True,
)
levels["arm_label"] = levels["arm"].map(arm_label)

# The same rows the difference figure shows, so the two can be read together.
plot_arm_levels_by_check(
    levels.merge(solid[["check", "property"]], on=["check", "property"]),
    series="arm_label",
    title="detailed and minimal, per property (near-complete pairings only)",
)

## 6. Layer S — did the arms return the right rows?

Layer 2 is conditional on layer 1, which is conditional on there being a row
at all. This is that bottom layer: how many row sets each arm got right,
missed entirely, or invented.

One stacked bar per arm, the two arms side by side. The stack shows the
total and each outcome's share of it; the pair shows the difference. The
cost is that `correct_row` outweighs `spurious_row` by roughly 200:1, so
the two error outcomes are slivers on top — read the exact counts from the
hover, or from the layer-S columns of the per-check figure below.

`plot-axis-units` appears four times because it evaluates four row sets —
`outputs` and three nested lists. Pooling them would add a sub-list's
spurious rows to the top-level list's.

**Each pair is `detailed` solid, `minimal` hatched.** Colour is spoken for by the outcome, so the variant is carried by the hatch — two shades of one green, touching, read as a single bar rather than a pair.

In [ ]:
layer_s = pd.concat(
    [
        layer_s_counts(
            load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
        )
        for check in checks
    ],
    ignore_index=True,
)
layer_s["arm_label"] = layer_s["arm"].map(arm_label)
# A check with one row set is named by itself; plot-axis-units has four.
layer_s["unit"] = [
    check if key == "outputs" else f"{check} · {key}"
    for check, key in zip(layer_s["check"], layer_s["list_key"])
]

plot_stacked_counts(
    layer_s,
    group="unit",
    category="outcome",
    series="arm_label",
    order=LAYER_S_ORDER,
    series_order=ARM_ORDER,
    ylabel="rows",
    title="Layer S row outcomes, summed over examples and replicates",
)

## 7. Layer 1 — did the arms judge applicability the same way?

Whether each leaf property was applicable, and whether that call was right.
`withheld_applicable` is the arm declining something it should have answered;
`spurious_applicable` is the reverse. Read these beside the non-response
counts above — an arm that withholds more has fewer instances left to score,
and a layer-2 mean over fewer, self-selected instances is the confound this
experiment is most exposed to.

Counts are summed over a check's leaf properties. That is a count, not a
score, so it is not the cross-property mean the rest of this notebook
refuses: `aggregate_layer1_counts` already sums exactly this.

Again stacked per arm, arms side by side. `correct_applicable` outweighs
`withheld_applicable` by between 14x and 208x depending on the check, so
the error bands are thin; the stack is here for the totals and the shares,
not for reading a count off the axis.

**Each pair is `detailed` solid, `minimal` hatched.** Colour is spoken for by the outcome, so the variant is carried by the hatch — two shades of one green, touching, read as a single bar rather than a pair.

In [ ]:
layer1 = pd.concat(
    [
        layer1_counts(
            load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
        )
        for check in checks
    ],
    ignore_index=True,
)
layer1["arm_label"] = layer1["arm"].map(arm_label)

plot_stacked_counts(
    layer1,
    group="check",
    category="layer1",
    series="arm_label",
    order=LAYER1_ORDER,
    series_order=ARM_ORDER,
    ylabel="instances",
    title="Layer 1 applicability calls, summed over properties and replicates",
)

## 8. One check, all three layers

The sections above take one layer across every check. This takes one check
across all three, because the layers are conditional on each other: layer 2
is scored only on instances layer 1 judged applicable, and layer 1 only
fires where layer S found a row. An arm that returns fewer rows has fewer
applicability calls to make, and an arm that withholds more has fewer
instances left for layer 2. Three separate figures make that chain something
you have to hold in your head.

Layer S and layer 1 stack, because their outcomes partition everything that
happened. Layer 2 does not: two arms' `mean_score` stacked would read as
their sum, which is not a quantity.

Arms sit side by side in every panel: `detailed` solid, `minimal` hatched.
In the stacked panels colour belongs to the outcome, so the variant is the
hatch. In layer 2 there is no outcome, so it takes the colour as well.

Set `DETAIL_CHECK` and re-run to inspect another check.

In [ ]:
DETAIL_CHECK = "error-bars-defined"

detail_runs = load_run_root(
    RUNS / DETAIL_CHECK, checklist=CHECKLIST, check=DETAIL_CHECK
)
detail_s = layer_s_counts(detail_runs)
detail_1 = layer1_counts(detail_runs)
detail_2 = replicate_spread(scores_frame(detail_runs))
for frame in (detail_s, detail_1, detail_2):
    frame["arm_label"] = frame["arm"].map(arm_label)

plot_check_layers(
    layer_s=detail_s,
    layer1=detail_1,
    layer2=detail_2,
    series="arm_label",
    series_order=ARM_ORDER,
    title=f"{DETAIL_CHECK} — layer S, layer 1 and layer 2",
).update_layout(height=600, width=1000)

## What this does not do

It does not decide whether the experiment succeeded. There is no overall
score, no significance test and no multiplicity correction, because what
counts as a real difference is the experiment's claim and belongs in
[`thinking/experiments/exp-01-skill-verbosity.md`](../../thinking/experiments/exp-01-skill-verbosity.md)
where the reasoning can be read.